In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import classification_report, precision_recall_curve, f1_score
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.preprocessing import TargetEncoder
from sklearn.ensemble import RandomForestClassifier
from category_encoders.target_encoder import TargetEncoder as CE_TargetEncoder
from scipy import stats
from sklearn.model_selection import RandomizedSearchCV

In [ ]:
files = [
    "/home/truphile/Downloads/classification data/data_descriptions.csv",
    "/home/truphile/Downloads/classification data/train.csv",
    "/home/truphile/Downloads/classification data/test.csv"
]

datasets = [pd.read_csv(file) for file in files]

data, train, test = datasets
data.head()

In [ ]:
# 1. PREPARE DATA & FEATURE ENGINEERING
# Clean TotalCharges (crucial before transformations)
train['TotalCharges'] = pd.to_numeric(train['TotalCharges'], errors='coerce').fillna(0)


In [ ]:
# Domain-specific feature engineering
train['AvgHistoricalCharge'] = train['TotalCharges'] / train['AccountAge'].replace(0, 1)
train['ChargeIncrease'] = train['MonthlyCharges'] - train['AvgHistoricalCharge']
train['TicketRate'] = train['SupportTicketsPerMonth'] / train['AccountAge'].replace(0, 1)


In [ ]:
# Drop CustomerID
X = train.drop(columns=['CustomerID', 'Churn'])
y = train["Churn"]

In [ ]:
# Train/Test Split
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

In [ ]:
# 2. PREPROCESSING (Target Encoding instead of OneHotEncoding)
# Target encoding replaces categories with the probability of churn for that category.
# This is incredibly powerful for tree models.
categorical_features = X_train.select_dtypes(exclude=np.number).columns.tolist()
numerical_features = X_train.select_dtypes(include=np.number).columns.tolist()

preprocessor = ColumnTransformer([
    ('num', 'passthrough', numerical_features), # Trees don't need PowerTransformer, but you can keep it
    ('cat', CE_TargetEncoder(), categorical_features)
])

In [ ]:
# 3. MODEL SETUP (Weighted RF instead of Balanced RF)
rf = make_pipeline(
    preprocessor,
    RandomForestClassifier(
        random_state=42,
        n_jobs=-1,
        class_weight='balanced_subsample' # FIX: Penalizes minority errors without deleting data
    )
)

In [ ]:
param_distributions = {
    'randomforestclassifier__n_estimators': stats.randint(100, 500), # Random int between 100 and 500
    'randomforestclassifier__max_depth': stats.randint(5, 40),      # Random int between 5 and 40
    'randomforestclassifier__min_samples_split': stats.randint(2, 20),
    'randomforestclassifier__min_samples_leaf': stats.randint(1, 15),
    'randomforestclassifier__max_features': ['sqrt', 'log2', 0.3, 0.5, 0.7] # Can still mix discrete values
}

random_search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_distributions,
    n_iter=30,             # ONLY TRY 30 RANDOM COMBINATIONS (vs 180 in grid)
    scoring='f1',
    cv=5,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

random_search.fit(X_train, y_train)
print("Best Coarse Params:", random_search.best_params_)
